In [ ]:
# 02 - SVM (TP3): SVM with a linear kernel and an RBF kernel, 3 folds, metrics, and MLflow
# Run 01_eda_data_wrangling.ipynb first: it creates ourdataset/train/normal/ and ourdataset/train/abnormal/
# The first time: run this cell (install MLflow, the SAME version as the mlflow-server container),
# then click "Restart" and run everything.
%pip install mlflow==2.22.4

In [ ]:
import pickle
from glob import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import IPython.display as ipd

from sklearn.model_selection import KFold            # to cut the dataset in 3 folds
from sklearn.preprocessing import StandardScaler     # to put all the features on the same scale
from sklearn.svm import SVC                          # the SVM classifier
from sklearn.pipeline import make_pipeline           # the scaler + the SVM in one object (final model)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

import mlflow
import mlflow.sklearn

In [ ]:
# my folders (the same as in 01)
project_folder = "D:/MLOps_Project/"                        # the .data files are saved here
ourdataset_folder = "D:/MLOps_Project/ourdataset/train/"


# the same 2 functions as in 01
def load_data(file_name):
    with open(file_name, "rb") as f:    # "rb" = read binary
        return pickle.load(f)           # gives back what we saved


def signaltonoise(a, axis=0, ddof=0):
    a = np.asanyarray(a)
    m = a.mean(axis)                        # mean (moyenne)
    sd = a.std(axis=axis, ddof=ddof)        # standard deviation (écart-type)
    return np.where(sd == 0, 0, m / sd)     # if std = 0 -> SNR = 0 (we can't divide by 0)

In [ ]:
# The SVM can't read a wav: it needs NUMBERS that describe each wav (= the "features").
# For each wav we take 22 numbers:
#   - 1 : its duration (TP1)
#   - 2 : its SNR (TP2)
#   - 3 to 22 : its Mel spectrogram (course 2) with 20 mel bands, and the MEAN of each band over time
#               (= how loud each band of frequencies is, on average. A murmur adds sound in some bands)
def get_wav_features(file):
    # the wavs don't all have the same sample rate (PhysioNet: 2000 Hz, Pascal: 4000 or 44100 Hz)
    # -> we load them ALL at 2000 Hz, so that the numbers mean the same thing for every wav
    signal, sample_rate = librosa.load(file, sr=2000)

    # 1. duration (TP1)
    duration = librosa.get_duration(y=signal, sr=sample_rate)

    # 2. SNR (TP2)
    norm = signal / max(np.amax(signal), -1 * np.amin(signal))
    snr = float(signaltonoise(norm))

    # 3 to 22. Mel spectrogram in dB (course 2)
    # n_fft=512 -> windows of 512 / 2000 = 0.25 s (2048 would be 1 s = a whole heartbeat, too long)
    S = librosa.feature.melspectrogram(y=signal, sr=sample_rate, n_fft=512, hop_length=128, n_mels=20)
    S_DB = librosa.power_to_db(S, ref=np.max)
    mel_means = np.mean(S_DB, axis=1)      # mean of each row (1 row = 1 mel band) -> 20 numbers

    return [duration, snr] + list(mel_means)    # 1 list of 22 numbers


def getFeatures(class_name):
    features = []
    for file in glob(ourdataset_folder + class_name + "/*.wav"):
        features.append(get_wav_features(file))    # 1 list of 22 numbers per wav

    file_name = project_folder + "features_" + class_name + "_train.data"
    with open(file_name, "wb") as f:
        pickle.dump(features, f)

    print(class_name, ":", len(features), "wavs x", len(features[0]), "features saved in", file_name)

In [ ]:
# reads the 3766 wavs, so it takes a few minutes. Run it once: after that, the .data files are enough
getFeatures("abnormal")    # -> features_abnormal_train.data
getFeatures("normal")      # -> features_normal_train.data

In [ ]:
features_normal = load_data(project_folder + "features_normal_train.data")
features_abnormal = load_data(project_folder + "features_abnormal_train.data")

# X = the features: 1 row per wav, 1 column per feature (the normal wavs first, then the abnormal ones)
X = np.array(features_normal + features_abnormal)

# y = the class of each wav = the answer the SVM must find: 0 = normal, 1 = abnormal
# ([0] * 3 gives [0, 0, 0])
y = np.array([0] * len(features_normal) + [1] * len(features_abnormal))

print("X:", X.shape, "->", X.shape[0], "wavs,", X.shape[1], "features")
print("y: normal (0) =", np.sum(y == 0), "| abnormal (1) =", np.sum(y == 1))

# the first 5 rows of X, as a table
feature_names = ["duration", "snr"] + ["mel_" + str(i) for i in range(1, 21)]
pd.DataFrame(X, columns=feature_names).head()

In [ ]:
# The 3 folds, with the example of the TP: 10 wavs numbered 1 to 10
# each fold: 1 third of the wavs = test (33%), the 2 other thirds = train (66%)
# each fold takes a DIFFERENT third for the test -> every wav is tested exactly once
def show_folds(kfold, items):
    fold = 1
    for train_index, test_index in kfold.split(items):
        print("  Fold", fold, ": test =", items[test_index], "  train =", items[train_index])
        fold = fold + 1


items = np.arange(1, 11)    # [ 1  2  3  4  5  6  7  8  9 10]

print("In order (like the example of the TP):")
show_folds(KFold(n_splits=3), items)
# 10 can't be cut in 3 equal parts, so the first test has 4 wavs (then wav 10 is tested too)

print("At random (random1 # random2: a different random third each time):")
show_folds(KFold(n_splits=3, shuffle=True, random_state=1), items)

In [ ]:
kernels = ["linear", "rbf"]

# 3 random folds. random_state=1 -> the SAME random folds every time we run it,
# so the 2 kernels are tested on exactly the same folds (fair comparison)
kfold = KFold(n_splits=3, shuffle=True, random_state=1)

results = []               # the metrics of each (kernel, fold)
confusion_matrices = []    # kept to draw them after

for kernel in kernels:
    print("==================== SVM with kernel =", kernel, "====================")
    fold = 1
    for train_index, test_index in kfold.split(X):
        # 1. split: 2 thirds of the wavs to train (66%), 1 third to test (33%)
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        # 2. same scale for every feature (mean 0, std 1): the SVM works with distances, and the
        #    duration (~20 s) would count much more than the SNR (~0). The mean and std are learned
        #    on the TRAIN only (fit), then used on the test too: the test wavs stay "never seen".
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # 3. train the SVM on the train wavs, then ask it the class of the test wavs
        #    class_weight="balanced": we have 3.5x more normal wavs than abnormal ones, so a mistake
        #    on an abnormal wav counts 3.5x more (if not, the SVM says "normal" too often)
        model = SVC(kernel=kernel, class_weight="balanced")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # 4. metrics: compare the predictions (y_pred) with the true classes (y_test)
        #    recall of abnormal = the % of sick hearts we found (sensitivity), recall of normal = specificity
        print("--- Fold", fold, ": train =", len(train_index), "wavs | test =", len(test_index), "wavs")
        print(classification_report(y_test, y_pred, target_names=["normal", "abnormal"]))

        results.append({
            "kernel": kernel,
            "fold": fold,
            "accuracy": accuracy_score(y_test, y_pred),     # % of good answers
            "precision": precision_score(y_test, y_pred),   # these 3: for the abnormal class (1)
            "recall": recall_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred),
        })
        confusion_matrices.append(confusion_matrix(y_test, y_pred))
        fold = fold + 1

In [ ]:
# all the results in a table: 1 row per (kernel, fold)
table = pd.DataFrame(results)
ipd.display(table.round(3))

# the mean of the 3 folds for each kernel (like the example: (0.7 + 0.8 + 0.9) / 3 = 0.8)
print("Mean of the 3 folds:")
ipd.display(table.groupby("kernel")[["accuracy", "precision", "recall", "f1"]].mean().round(3))

In [ ]:
def PlotConfusionMatrix(cm, title):
    # cm[i][j] = number of test wavs of class i (true) that the SVM put in class j (predicted)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["normal", "abnormal"], yticklabels=["normal", "abnormal"])
    plt.title(title)
    plt.xlabel("Predicted class")
    plt.ylabel("True class")


# ONE figure: 2 rows (linear, rbf) x 4 columns (fold 1, fold 2, fold 3, accuracy)
plt.figure(figsize=(22, 9))

# the 6 confusion matrices (in the order of the loop: linear fold 1, 2, 3, then rbf fold 1, 2, 3)
places = [1, 2, 3, 5, 6, 7]    # column 4 (places 4 and 8) is kept for the bar plot
for i in range(6):
    r = results[i]
    plt.subplot(2, 4, places[i])
    PlotConfusionMatrix(confusion_matrices[i],
                        r["kernel"] + " - Fold " + str(r["fold"]) + " - accuracy=" + str(round(r["accuracy"], 2)))

# column 4 (places 4 AND 8): the accuracy of each fold, linear vs rbf
accuracy = table["accuracy"].values     # 6 values: linear fold 1, 2, 3, then rbf fold 1, 2, 3
x = np.arange(3)                        # 0, 1, 2 = fold 1, 2, 3

plt.subplot(2, 4, (4, 8))
plt.bar(x - 0.2, accuracy[0:3], width=0.4, label="linear")
plt.bar(x + 0.2, accuracy[3:6], width=0.4, label="rbf")
plt.xticks(x, ["Fold 1", "Fold 2", "Fold 3"])
plt.ylim(0, 1)
plt.ylabel("Accuracy")
plt.title("Accuracy of each fold: linear vs rbf")
plt.legend(loc="lower right")

plt.tight_layout()
plt.savefig(project_folder + "tp3_svm_figure.png")    # also saved as an image file, to send it to MLflow
plt.show()

In [ ]:
# ---- MLflow (Service 6) ----
# our MLflow server = the Docker container "mlflow-server" (it must be running), on port 5000
mlflow.set_tracking_uri("http://localhost:5000")

# ONE experiment for the 5 models of the project (SVM, Decision Tree, CNN, ResNet, Transformer),
# so that MLflow compares them all in the same table and charts (created the first time)
mlflow.set_experiment("audio_benchmark")

In [ ]:
# one MLflow "run" per kernel = one line in the MLflow table
for kernel in kernels:
    rows = table[table["kernel"] == kernel]    # the 3 lines of the results table for this kernel (fold 1, 2, 3)

    with mlflow.start_run(run_name="svm_" + kernel):
        # 1. params = the settings we chose
        mlflow.log_param("model", "SVM")
        mlflow.log_param("kernel", kernel)
        mlflow.log_param("class_weight", "balanced")
        mlflow.log_param("n_folds", 3)
        mlflow.log_param("features", "duration + snr + 20 mel means")

        # 2. metrics = the results: the mean of the 3 folds (the columns of the MLflow table)
        mlflow.log_metric("mean_accuracy", rows["accuracy"].mean())
        mlflow.log_metric("mean_precision", rows["precision"].mean())
        mlflow.log_metric("mean_recall", rows["recall"].mean())    # recall of abnormal: the project picks the best model with it
        mlflow.log_metric("mean_f1", rows["f1"].mean())

        # ... and the result of each fold (step = the fold number -> MLflow draws it as a small chart)
        for r in results:
            if r["kernel"] == kernel:
                mlflow.log_metric("fold_accuracy", r["accuracy"], step=r["fold"])
                mlflow.log_metric("fold_recall", r["recall"], step=r["fold"])

        # 3. artifacts = files: the TP3 figure (saved as a png by the figure cell)
        mlflow.log_artifact(project_folder + "tp3_svm_figure.png")

        # 4. the model. The 3 folds were only to MEASURE how good the SVM is.
        #    The model we keep is trained on ALL the wavs. make_pipeline = the scaler + the SVM in ONE object,
        #    so later we can give it the features of a new wav directly (like the audio engine will do)
        final_model = make_pipeline(StandardScaler(), SVC(kernel=kernel, class_weight="balanced"))
        final_model.fit(X, y)
        mlflow.sklearn.log_model(final_model, "model", input_example=X[:5])

# the best kernel for the project = the highest recall on the abnormal class (the sick hearts)
mean_recall = table.groupby("kernel")["recall"].mean()
print(mean_recall.round(3))
print("Best kernel (highest recall of abnormal):", mean_recall.idxmax())